# VaR-Filtered Mean Reversion with Holdout Analysis

This long/cash QQQ backtest combines a rolling z-score with a value-at-risk gate and compares results with SPY. Yahoo Finance supplies adjusted historical prices. The annual risk-free input is the mean observed three-month Treasury yield in the training period and is held fixed for holdout evaluation. [Federal Reserve three-month constant maturity yield via FRED](https://fred.stlouisfed.org/series/DGS3MO)


In [ ]:
"""
VOLATILITY-ADJUSTED MEAN REVERSION STRATEGY (VaR-FILTERED)
===========================================================

STRATEGY OVERVIEW:
-----------------
This strategy exploits mean reversion opportunities ONLY when downside risk is low.
The key insight is that mean reversion works better in stable environments - when
volatility spikes, mean reversion can fail catastrophically.


WHY THIS WORKS:
--------------
1. MEAN REVERSION: Assets tend to revert to their mean after extreme deviations
2. VaR FILTER: Only trade when tail risk is low (prevents catastrophic losses)
3. Z-SCORE: Identifies statistical extremes (oversold conditions)
4. RISK CONTROL: VaR acts as volatility regime filter
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from _market_data import download_ohlcv
import warnings
from datetime import datetime, timedelta
from typing import Dict, Tuple, Optional, List
from dataclasses import dataclass

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class StrategyConfig:
    """
    Centralized configuration for all strategy parameters.

    CRITICAL: All parameters are configurable here - no hardcoding.
    This allows easy optimization and sensitivity analysis.
    """

    # API Configuration

    # Date Range (5 years ending 2024-12-31)
    END_DATE = "2024-12-31"
    START_DATE = "2019-12-31"  # 5 years before end date

    # Trading Parameters
    INITIAL_CAPITAL = 10000.0  # Starting capital
    COMMISSION_PER_TRADE = 1.0  # Fixed $1 commission per trade

    # Signal Generation Parameters
    ZSCORE_WINDOW = 20  # Rolling window for z-score calculation
    ZSCORE_THRESHOLD = -1.0  # Oversold threshold (< -1 = oversold)
    VAR_WINDOW = 20  # Rolling window for VaR calculation
    VAR_PERCENTILE = 10  # 10th percentile for VaR (10% worst cases)
    VAR_THRESHOLD = 0.02  # Maximum acceptable VaR (2% threshold)

    # Trading Ticker (mean reversion candidate)
    TICKER = "QQQ"  # Nasdaq-100 ETF - liquid and volatile

    # Benchmark
    BENCHMARK = "SPY"  # S&P 500 ETF

    # Risk-Free Rate (three-month Treasury yield proxy)
    RISK_FREE_RATE: float | None = None

    # Out-of-Sample Validation
    TRAIN_END_DATE = "2022-06-30"  # 70% train / 30% test split

    # Robustness Analysis
    ZSCORE_RANGE = [15, 20, 25]  # Test different z-score windows
    ZSCORE_THRESH_RANGE = [-0.5, -1.0, -1.5]  # Test different thresholds
    VAR_THRESH_RANGE = [0.015, 0.02, 0.025]  # Test different VaR thresholds

    # Performance Analysis
    ROLLING_SHARPE_WINDOW = 126  # 6 months for rolling Sharpe


# ============================================================================
# DATA ACQUISITION
# ============================================================================

class DataFetcher:
    """
    Loads adjusted market prices from Yahoo Finance and Treasury yields from FRED.

    CRITICAL: No forward-filling or back-filling of missing data.
    We use ONLY the data that exists to maintain realism.
    """


    def fetch_historical_data(self, ticker: str, start_date: str, end_date: str) -> pd.DataFrame:
        buffer_start = pd.Timestamp(start_date) - timedelta(days=30)
        prices = download_ohlcv(ticker, buffer_start, end_date, auto_adjust=True)
        if len(prices) < 50:
            raise ValueError(f"Insufficient daily observations for {ticker}.")
        return prices.rename_axis("Date").reset_index()

    def fetch_risk_free_rate(self, start_date: str, end_date: str) -> pd.DataFrame:
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS3MO&cosd={start_date}&coed={end_date}"
        rates = pd.read_csv(url, na_values=["."])
        dates = pd.to_datetime(rates.iloc[:, 0], errors="raise")
        yields = pd.to_numeric(rates["DGS3MO"], errors="raise") / 100.0
        result = pd.DataFrame({"date": dates, "rf_rate": yields}).dropna().sort_values("date")
        result = result.loc[result["date"].between(start_date, end_date)]
        if result.empty:
            raise RuntimeError("FRED returned no three-month Treasury yields in the requested range.")
        return result.reset_index(drop=True)


# ============================================================================
# FEATURE ENGINEERING (NO LOOK-AHEAD BIAS)
# ============================================================================

class FeatureEngine:
    """
    Calculates technical features using STRICT T-1 methodology.

    CRITICAL PRINCIPLE: All calculations for date T use ONLY data up to T-1.
    Features are lagged by 1 day before use in signal generation.
    """

    @staticmethod
    def calculate_log_returns(prices: pd.Series) -> pd.Series:
        """
        Calculate log returns.

        Log returns are time-additive and better for statistical analysis:
        - log(P[t]/P[t-1]) = log(P[t]) - log(P[t-1])
        - More symmetric than simple returns
        - Better for mean reversion analysis

        Parameters:
        -----------
        prices : pd.Series
            Price series

        Returns:
        --------
        pd.Series of log returns
        """
        return np.log(prices / prices.shift(1))

    @staticmethod
    def calculate_zscore(returns: pd.Series, window: int) -> pd.Series:
        """
        Calculate rolling z-score of returns.

        Z-SCORE INTERPRETATION:
        - z < -2: Extremely oversold (2+ std devs below mean)
        - z < -1: Oversold (1+ std dev below mean)
        - -1 < z < 1: Normal range
        - z > 1: Overbought (1+ std dev above mean)
        - z > 2: Extremely overbought

        MEAN REVERSION LOGIC:
        When z-score is very negative, asset is statistically cheap
        and likely to revert upward toward its mean.

        Parameters:
        -----------
        returns : pd.Series
            Return series
        window : int
            Rolling window size

        Returns:
        --------
        pd.Series of z-scores
        """
        # Calculate rolling mean and std
        rolling_mean = returns.rolling(window=window, min_periods=window).mean()
        rolling_std = returns.rolling(window=window, min_periods=window).std()

        # Z-score = (current - mean) / std
        # Clip std to avoid division by near-zero
        rolling_std = rolling_std.clip(lower=1e-6)

        zscore = (returns - rolling_mean) / rolling_std

        return zscore

    @staticmethod
    def calculate_historical_var(returns: pd.Series, window: int,
                                percentile: float) -> pd.Series:
        """
        Calculate historical Value-at-Risk (VaR).

        VaR INTERPRETATION:
        VaR(10%) = 0.02 means: "In the worst 10% of days, we lose at least 2%"

        WHY VAR AS VOLATILITY FILTER:
        - VaR captures tail risk (worst-case scenarios)
        - Low VaR = stable environment = mean reversion works better
        - High VaR = turbulent market = mean reversion can fail
        - More robust than simple volatility (std) for risk filtering

        CALCULATION METHOD:
        Historical VaR uses empirical distribution (no parametric assumptions)
        - Take rolling window of past returns
        - Calculate specified percentile (e.g., 10th)
        - This is the VaR estimate

        Parameters:
        -----------
        returns : pd.Series
            Return series
        window : int
            Rolling window size
        percentile : float
            Percentile for VaR (e.g., 10 for 10th percentile)

        Returns:
        --------
        pd.Series of VaR values (positive = loss)
        """
        # Calculate rolling percentile of returns
        # Lower percentile = worse outcomes (left tail)
        var_series = returns.rolling(
            window=window,
            min_periods=window
        ).quantile(percentile / 100.0)

        # Convert to positive loss values (easier interpretation)
        # VaR is typically expressed as positive loss amount
        var_series = -var_series  # Negative return = positive loss

        return var_series

    @staticmethod
    def prepare_features(df: pd.DataFrame, config: StrategyConfig) -> pd.DataFrame:
        """
        Prepare all features with proper lagging to avoid look-ahead bias.

        EXECUTION FLOW FOR FEATURES:
        1. Calculate raw features from OHLCV data
        2. LAG all features by 1 day (critical!)
        3. At close of day T-1, we know lagged features for T-1
        4. We use T-1 lagged features to generate signal for execution at T

        This ensures that signals for day T use only information
        available at close of day T-1.

        Parameters:
        -----------
        df : pd.DataFrame
            OHLCV data
        config : StrategyConfig
            Strategy configuration

        Returns:
        --------
        pd.DataFrame with lagged features
        """
        data = df.copy()

        # Calculate log returns (for z-score)
        data["LogReturns"] = FeatureEngine.calculate_log_returns(data["Close"])

        # Calculate z-score (identifies oversold/overbought)
        data["ZScore"] = FeatureEngine.calculate_zscore(
            data["LogReturns"],
            config.ZSCORE_WINDOW
        )

        # Calculate VaR (tail risk filter)
        data["VaR"] = FeatureEngine.calculate_historical_var(
            data["LogReturns"],
            config.VAR_WINDOW,
            config.VAR_PERCENTILE
        )

        # CRITICAL: LAG all features by 1 day
        # At close of day T-1, we know features calculated using data up to T-1
        # We lag these features so that on day T, we access T-1 features
        data["ZScore_Lag"] = data["ZScore"].shift(1)
        data["VaR_Lag"] = data["VaR"].shift(1)

        # Also calculate simple returns for portfolio valuation
        data["Returns"] = data["Close"].pct_change()

        return data


# ============================================================================
# SIGNAL GENERATOR (T-1 → T LOGIC)
# ============================================================================

class SignalGenerator:
    """
    Generates trading signals using T-1 features.

    SIGNAL LOGIC:
    - Long (signal = 1) when:
      1. Z-score < threshold (oversold)
      2. VaR < threshold (low tail risk)
    - Cash (signal = 0) otherwise

    This combines mean reversion opportunity (oversold) with risk control (low VaR).
    """

    def __init__(self, config: StrategyConfig):
        self.config = config

    def generate_signals(self, data: pd.DataFrame) -> pd.DataFrame:
        """
        Generate trading signals using lagged features.

        EXECUTION FLOW:
        - Signal generated at Close[T-1] using lagged features
        - Signal stored in row T-1
        - Signal executed at Open[T]

        Parameters:
        -----------
        data : pd.DataFrame
            Data with lagged features

        Returns:
        --------
        pd.DataFrame with signals
        """
        df = data.copy()

        # Initialize signal column
        df["Signal"] = 0

        # LONG SIGNAL CONDITIONS:
        # 1. Oversold: Z-score below threshold (< -1)
        # 2. Low Risk: VaR below threshold (< 0.02)
        condition_oversold = df["ZScore_Lag"] < self.config.ZSCORE_THRESHOLD
        condition_low_risk = df["VaR_Lag"] < self.config.VAR_THRESHOLD

        # Generate long signal when BOTH conditions met
        df.loc[condition_oversold & condition_low_risk, "Signal"] = 1

        # Track signal changes for trade counting
        df["Signal_Change"] = df["Signal"].diff()

        return df


# ============================================================================
# BACKTEST ENGINE (REALISTIC EXECUTION)
# ============================================================================

class BacktestEngine:
    """
    Realistic backtesting engine with proper execution logic.

    EXECUTION FLOW:
    Day T-1 (Signal Day):
      - Observe Close[T-1]
      - Calculate features using data up to T-1
      - Generate signal

    Day T (Execution Day):
      - Execute at Open[T]
      - Pay commission
      - Update positions

    Day T (Close):
      - Value portfolio at Close[T]
      - Calculate returns: Close[T]/Close[T-1] - 1

    COMMISSIONS:
    - $1 per trade (entry)
    - $1 per trade (exit)
    - Reduces realized returns
    - Important for realism (eliminates high-frequency artifacts)
    """

    def __init__(self, config: StrategyConfig):
        self.config = config
        self.trades = []

    def run_backtest(self, data: pd.DataFrame) -> Dict:
        """
        Execute backtest with realistic execution.

        Returns:
        --------
        Dict with complete backtest results
        """
        print("\n" + "="*80)
        print("EXECUTING BACKTEST (T-1 → T LOGIC)")
        print("="*80)

        df = data.copy()

        # Initialize tracking
        df["Position"] = 0  # Current position (0 or 1)
        df["Cash"] = self.config.INITIAL_CAPITAL
        df["Holdings"] = 0.0
        df["Portfolio_Value"] = self.config.INITIAL_CAPITAL
        df["Strategy_Returns"] = 0.0

        # Portfolio state
        cash = self.config.INITIAL_CAPITAL
        shares = 0
        position = 0

        # Execution loop
        for i in range(1, len(df)):
            current_idx = df.index[i]
            prev_idx = df.index[i-1]

            # Get previous signal (generated at T-1 Close)
            signal = df.loc[prev_idx, "Signal"]

            # Current prices
            open_price = df.loc[current_idx, "Open"]
            close_price = df.loc[current_idx, "Close"]

            # Check for missing prices
            if pd.isna(open_price) or pd.isna(close_price):
                df.loc[current_idx, "Position"] = position
                df.loc[current_idx, "Cash"] = cash
                df.loc[current_idx, "Holdings"] = shares * close_price if not pd.isna(close_price) else 0
                df.loc[current_idx, "Portfolio_Value"] = cash + (shares * close_price if not pd.isna(close_price) else 0)
                continue

            # === EXECUTION AT OPEN[T] ===

            # CASE 1: Enter Long Position
            if signal == 1 and position == 0:
                # Calculate shares we can buy
                # Available cash after commission
                available = cash - self.config.COMMISSION_PER_TRADE

                if available > 0:
                    shares = available / open_price
                    cost = shares * open_price + self.config.COMMISSION_PER_TRADE
                    cash -= cost
                    position = 1

                    self.trades.append({
                        "Date": df.loc[current_idx, "Date"],
                        "Action": "BUY",
                        "Price": open_price,
                        "Shares": shares,
                        "Commission": self.config.COMMISSION_PER_TRADE,
                        "Cash_After": cash
                    })

            # CASE 2: Exit Long Position
            elif signal == 0 and position == 1:
                # Sell shares at Open[T]
                proceeds = shares * open_price - self.config.COMMISSION_PER_TRADE
                cash += proceeds

                self.trades.append({
                    "Date": df.loc[current_idx, "Date"],
                    "Action": "SELL",
                    "Price": open_price,
                    "Shares": shares,
                    "Commission": self.config.COMMISSION_PER_TRADE,
                    "Cash_After": cash
                })

                shares = 0
                position = 0

            # === VALUATION AT CLOSE[T] ===

            holdings_value = shares * close_price
            portfolio_value = cash + holdings_value

            # Calculate returns (Close[T] / Close[T-1] - 1)
            if i > 0:
                prev_portfolio_value = df.loc[prev_idx, "Portfolio_Value"]
                if prev_portfolio_value > 0:
                    strategy_return = (portfolio_value / prev_portfolio_value) - 1
                else:
                    strategy_return = 0
            else:
                strategy_return = 0

            # Store results
            df.loc[current_idx, "Position"] = position
            df.loc[current_idx, "Cash"] = cash
            df.loc[current_idx, "Holdings"] = holdings_value
            df.loc[current_idx, "Portfolio_Value"] = portfolio_value
            df.loc[current_idx, "Strategy_Returns"] = strategy_return

        # Calculate cumulative returns
        df["Strategy_Cumulative"] = (1 + df["Strategy_Returns"]).cumprod()

        print(f"\n✓ Backtest completed")
        print(f"  Total trades: {len(self.trades)}")
        print(f"  Final portfolio value: ${df['Portfolio_Value'].iloc[-1]:,.2f}")
        print(f"  Total return: {(df['Portfolio_Value'].iloc[-1] / self.config.INITIAL_CAPITAL - 1):.2%}")

        return {
            "data": df,
            "trades": self.trades,
            "initial_capital": self.config.INITIAL_CAPITAL,
            "final_value": df["Portfolio_Value"].iloc[-1]
        }

    def add_benchmark(self, results: Dict, benchmark_data: pd.DataFrame) -> Dict:
        """
        Add benchmark buy-and-hold performance.

        BENCHMARK EXECUTION:
        - Buy at first Open with commission
        - Hold until end
        - Sell at last Open with commission

        Parameters:
        -----------
        results : Dict
            Backtest results
        benchmark_data : pd.DataFrame
            Benchmark price data

        Returns:
        --------
        Dict with benchmark added
        """
        df = results["data"]

        # Merge benchmark data
        bench = benchmark_data[["Date", "Open", "Close"]].rename(
            columns={"Open": "Bench_Open", "Close": "Bench_Close"}
        )
        df = df.merge(bench, on="Date", how="left")

        # Calculate benchmark with commissions
        first_open = df["Bench_Open"].iloc[0]

        if pd.notna(first_open) and first_open > 0:
            # Buy shares at first Open (after commission)
            available = self.config.INITIAL_CAPITAL - self.config.COMMISSION_PER_TRADE
            shares = available / first_open

            # Calculate value at each Close
            df["Bench_Value"] = shares * df["Bench_Close"]

            # Apply exit commission at end
            last_idx = df.index[-1]
            last_open = df.loc[last_idx, "Bench_Open"]
            if pd.notna(last_open):
                final_value = shares * last_open - self.config.COMMISSION_PER_TRADE
                df.loc[last_idx, "Bench_Value"] = final_value

            # Calculate benchmark returns
            df["Bench_Returns"] = df["Bench_Close"].pct_change()
            df["Bench_Cumulative"] = (1 + df["Bench_Returns"]).cumprod()
        else:
            df["Bench_Value"] = self.config.INITIAL_CAPITAL
            df["Bench_Returns"] = 0
            df["Bench_Cumulative"] = 1

        results["data"] = df

        print(f"✓ Benchmark added: Final value ${df['Bench_Value'].iloc[-1]:,.2f}")

        return results


# ============================================================================
# OUT-OF-SAMPLE VALIDATION
# ============================================================================

class OutOfSampleValidator:
    """
    Validates strategy performance on unseen data.

    CRITICAL IMPROVEMENT: Preventing Overfitting
    ============================================
    The biggest risk in backtesting is overfitting:
    - Parameters optimized on full dataset
    - Strategy "memorizes" historical patterns
    - Poor performance on new data

    SOLUTION: Train/Test Split
    - Train: Optimize parameters (in-sample)
    - Test: Validate performance (out-of-sample)
    - Test data NEVER used for parameter selection

    This mimics real trading: parameters chosen on past,
    applied to future (unknown) data.
    """

    @staticmethod
    def split_data(data: pd.DataFrame, train_end_date: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Split data into train and test sets.

        METHODOLOGY:
        - Train: Start → train_end_date (typically 70% of data)
        - Test: train_end_date → End (typically 30% of data)
        - Chronological split (no random sampling)

        Parameters:
        -----------
        data : pd.DataFrame
            Full dataset
        train_end_date : str
            Last date of training period (YYYY-MM-DD)

        Returns:
        --------
        Tuple of (train_df, test_df)
        """
        train_end = pd.to_datetime(train_end_date)

        train_data = data[data["Date"] <= train_end].copy()
        test_data = data[data["Date"] > train_end].copy()

        print(f"\n" + "="*80)
        print("OUT-OF-SAMPLE VALIDATION: DATA SPLIT")
        print("="*80)
        print(f"\nTrain Period:")
        print(f"  Dates: {train_data['Date'].min().date()} to {train_data['Date'].max().date()}")
        print(f"  Days: {len(train_data)}")
        print(f"  Percentage: {len(train_data)/(len(train_data)+len(test_data))*100:.1f}%")

        print(f"\nTest Period (Out-of-Sample):")
        print(f"  Dates: {test_data['Date'].min().date()} to {test_data['Date'].max().date()}")
        print(f"  Days: {len(test_data)}")
        print(f"  Percentage: {len(test_data)/(len(train_data)+len(test_data))*100:.1f}%")

        return train_data, test_data

    @staticmethod
    def compare_performance(train_metrics: Dict, test_metrics: Dict):
        """
        Compare in-sample vs out-of-sample performance.

        KEY VALIDATION CRITERIA:
        - Test Sharpe > 0: Strategy has predictive power
        - Test Sharpe ≈ Train Sharpe: Not overfit
        - Test Sharpe << Train Sharpe: Likely overfit
        - Test drawdown < Train drawdown: Robust

        Parameters:
        -----------
        train_metrics : Dict
            In-sample metrics
        test_metrics : Dict
            Out-of-sample metrics
        """
        print(f"\n" + "="*80)
        print("OUT-OF-SAMPLE VALIDATION: PERFORMANCE COMPARISON")
        print("="*80)

        print(f"\n{'Metric':<25} {'Train (In-Sample)':<20} {'Test (Out-Sample)':<20} {'Degradation':<15}")
        print("-"*80)

        metrics_to_compare = [
            ("CAGR", "%"),
            ("Sharpe Ratio", "ratio"),
            ("Sortino Ratio", "ratio"),
            ("Max Drawdown", "%"),
            ("Hit Ratio", "%")
        ]

        for metric, fmt in metrics_to_compare:
            train_val = train_metrics["Strategy"][metric]
            test_val = test_metrics["Strategy"][metric]

            if fmt == "%":
                train_str = f"{train_val:>8.2%}"
                test_str = f"{test_val:>8.2%}"

                # Calculate degradation
                if train_val != 0:
                    deg = (test_val - train_val) / abs(train_val)
                    deg_str = f"{deg:>+8.1%}"
                else:
                    deg_str = "N/A"
            else:
                train_str = f"{train_val:>8.2f}"
                test_str = f"{test_val:>8.2f}"

                if train_val != 0:
                    deg = (test_val - train_val) / abs(train_val)
                    deg_str = f"{deg:>+8.1%}"
                else:
                    deg_str = "N/A"

            print(f"{metric:<25} {train_str:<20} {test_str:<20} {deg_str:<15}")

        # Overfitting assessment
        train_sharpe = train_metrics["Strategy"]["Sharpe Ratio"]
        test_sharpe = test_metrics["Strategy"]["Sharpe Ratio"]

        print(f"\n" + "="*80)
        print("OVERFITTING ASSESSMENT:")
        print("="*80)

        if test_sharpe <= 0:
            print("⚠ WARNING: Negative out-of-sample Sharpe - Strategy has NO predictive power")
            print("  Recommendation: DO NOT TRADE this strategy")
        elif test_sharpe < train_sharpe * 0.5:
            print("⚠ WARNING: Severe performance degradation (>50% Sharpe drop)")
            print("  Likely OVERFIT to training data")
            print("  Recommendation: Revise parameters or strategy logic")
        elif test_sharpe < train_sharpe * 0.7:
            print("⚠ CAUTION: Moderate performance degradation (30-50% Sharpe drop)")
            print("  Some overfitting present")
            print("  Recommendation: Test with more robust parameters")
        else:
            print("✓ GOOD: Out-of-sample performance holds up well")
            print("  Strategy appears robust and not overfit")
            print("  Recommendation: Proceed with live testing")


# ============================================================================
# ROBUSTNESS ANALYSIS
# ============================================================================

class RobustnessAnalyzer:
    """
    Tests strategy robustness across parameters and periods.

    CRITICAL IMPROVEMENT: Parameter Stability
    =========================================
    A robust strategy should:
    - Work across different parameters (not knife-edge)
    - Work in different market conditions
    - Not depend on perfect parameter selection

    METHODOLOGY:
    - Grid search across parameter ranges
    - Test multiple time periods
    - Identify stable parameter regions
    - Avoid data mining (use sensible ranges)
    """

    def __init__(self, config: StrategyConfig):
        self.config = config

    def parameter_sensitivity_analysis(self,
                                      data: pd.DataFrame,
                                      benchmark_data: pd.DataFrame) -> pd.DataFrame:
        """
        Test strategy across parameter grid.

        PARAMETERS TESTED:
        - Z-score window: 15, 20, 25 days
        - Z-score threshold: -0.5, -1.0, -1.5
        - VaR threshold: 1.5%, 2.0%, 2.5%

        Total: 3 × 3 × 3 = 27 combinations

        INTERPRETATION:
        - High Sharpe across many combos: Robust
        - High Sharpe only for 1-2 combos: Fragile/Overfit
        - Similar results across parameters: Stable

        Returns:
        --------
        pd.DataFrame with results for each parameter combination
        """
        print(f"\n" + "="*80)
        print("ROBUSTNESS ANALYSIS: PARAMETER SENSITIVITY")
        print("="*80)

        results_list = []

        total_combos = (len(self.config.ZSCORE_RANGE) *
                       len(self.config.ZSCORE_THRESH_RANGE) *
                       len(self.config.VAR_THRESH_RANGE))

        combo_num = 0

        for zscore_window in self.config.ZSCORE_RANGE:
            for zscore_thresh in self.config.ZSCORE_THRESH_RANGE:
                for var_thresh in self.config.VAR_THRESH_RANGE:
                    combo_num += 1

                    print(f"\nTesting combo {combo_num}/{total_combos}: "
                          f"Z-window={zscore_window}, Z-thresh={zscore_thresh:.1f}, "
                          f"VaR-thresh={var_thresh:.3f}")

                    # Create temporary config
                    temp_config = StrategyConfig()
                    temp_config.RISK_FREE_RATE = self.config.RISK_FREE_RATE
                    temp_config.ZSCORE_WINDOW = zscore_window
                    temp_config.ZSCORE_THRESHOLD = zscore_thresh
                    temp_config.VAR_THRESHOLD = var_thresh

                    try:
                        # Run backtest with these parameters
                        data_with_features = FeatureEngine.prepare_features(
                            data.copy(), temp_config
                        )

                        signal_gen = SignalGenerator(temp_config)
                        data_with_signals = signal_gen.generate_signals(data_with_features)

                        engine = BacktestEngine(temp_config)
                        results = engine.run_backtest(data_with_signals)
                        results = engine.add_benchmark(results, benchmark_data)

                        metrics = PerformanceAnalyzer.calculate_metrics(
                            results, temp_config.RISK_FREE_RATE
                        )

                        # Store results
                        results_list.append({
                            "ZScore_Window": zscore_window,
                            "ZScore_Threshold": zscore_thresh,
                            "VaR_Threshold": var_thresh,
                            "CAGR": metrics["Strategy"]["CAGR"],
                            "Sharpe": metrics["Strategy"]["Sharpe Ratio"],
                            "Sortino": metrics["Strategy"]["Sortino Ratio"],
                            "Max_DD": metrics["Strategy"]["Max Drawdown"],
                            "Trades": metrics["Strategy"]["Total Trades"],
                            "Hit_Ratio": metrics["Strategy"]["Hit Ratio"]
                        })

                    except Exception as e:
                        print(f"  Error: {str(e)}")
                        continue

        # Create results dataframe
        results_df = pd.DataFrame(results_list)

        # Summary statistics
        print(f"\n" + "="*80)
        print("ROBUSTNESS SUMMARY:")
        print("="*80)

        print(f"\nSharpe Ratio Distribution:")
        print(f"  Mean:   {results_df['Sharpe'].mean():.2f}")
        print(f"  Median: {results_df['Sharpe'].median():.2f}")
        print(f"  Std:    {results_df['Sharpe'].std():.2f}")
        print(f"  Min:    {results_df['Sharpe'].min():.2f}")
        print(f"  Max:    {results_df['Sharpe'].max():.2f}")

        print(f"\nCAGR Distribution:")
        print(f"  Mean:   {results_df['CAGR'].mean():.2%}")
        print(f"  Median: {results_df['CAGR'].median():.2%}")
        print(f"  Std:    {results_df['CAGR'].std():.2%}")

        # Best performing parameters
        best_idx = results_df['Sharpe'].idxmax()
        best_params = results_df.loc[best_idx]

        print(f"\nBest Parameters (by Sharpe):")
        print(f"  Z-Score Window: {best_params['ZScore_Window']}")
        print(f"  Z-Score Threshold: {best_params['ZScore_Threshold']:.1f}")
        print(f"  VaR Threshold: {best_params['VaR_Threshold']:.3f}")
        print(f"  Sharpe Ratio: {best_params['Sharpe']:.2f}")
        print(f"  CAGR: {best_params['CAGR']:.2%}")

        # Robustness assessment
        sharpe_std = results_df['Sharpe'].std()
        sharpe_mean = results_df['Sharpe'].mean()
        coefficient_of_variation = sharpe_std / sharpe_mean if sharpe_mean > 0 else np.inf

        print(f"\n" + "="*80)
        print("ROBUSTNESS ASSESSMENT:")
        print("="*80)

        if coefficient_of_variation < 0.3:
            print("✓ EXCELLENT: Very stable performance across parameters")
            print("  Strategy is robust and not parameter-dependent")
        elif coefficient_of_variation < 0.5:
            print("✓ GOOD: Reasonably stable performance")
            print("  Strategy shows moderate robustness")
        else:
            print("⚠ WARNING: High parameter sensitivity")
            print("  Strategy may be overfit or fragile")
            print("  Recommendation: Use conservative parameters")

        return results_df

    def visualize_parameter_sensitivity(self, results_df: pd.DataFrame):
        """
        Visualize parameter sensitivity analysis.

        Creates heatmaps showing how Sharpe ratio varies
        with different parameter combinations.
        """
        fig, axes = plt.subplots(1, 3, figsize=(20, 6))

        # Heatmap 1: Z-Score Window vs Threshold (averaging over VaR)
        pivot1 = results_df.pivot_table(
            values='Sharpe',
            index='ZScore_Threshold',
            columns='ZScore_Window',
            aggfunc='mean'
        )

        sns.heatmap(pivot1, annot=True, fmt='.2f', cmap='RdYlGn',
                   center=0, ax=axes[0], cbar_kws={'label': 'Sharpe Ratio'})
        axes[0].set_title('Sharpe: Z-Score Window vs Threshold',
                         fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Z-Score Window (days)')
        axes[0].set_ylabel('Z-Score Threshold')

        # Heatmap 2: VaR Threshold vs Z-Score Window (averaging over Z-thresh)
        pivot2 = results_df.pivot_table(
            values='Sharpe',
            index='VaR_Threshold',
            columns='ZScore_Window',
            aggfunc='mean'
        )

        sns.heatmap(pivot2, annot=True, fmt='.2f', cmap='RdYlGn',
                   center=0, ax=axes[1], cbar_kws={'label': 'Sharpe Ratio'})
        axes[1].set_title('Sharpe: VaR Threshold vs Z-Score Window',
                         fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Z-Score Window (days)')
        axes[1].set_ylabel('VaR Threshold')

        # Heatmap 3: VaR Threshold vs Z-Score Threshold (averaging over window)
        pivot3 = results_df.pivot_table(
            values='Sharpe',
            index='ZScore_Threshold',
            columns='VaR_Threshold',
            aggfunc='mean'
        )

        sns.heatmap(pivot3, annot=True, fmt='.2f', cmap='RdYlGn',
                   center=0, ax=axes[2], cbar_kws={'label': 'Sharpe Ratio'})
        axes[2].set_title('Sharpe: Z-Score Threshold vs VaR Threshold',
                         fontsize=12, fontweight='bold')
        axes[2].set_xlabel('VaR Threshold')
        axes[2].set_ylabel('Z-Score Threshold')

        plt.tight_layout()
        plt.show()


# ============================================================================
# PERFORMANCE ANALYSIS
# ============================================================================

class PerformanceAnalyzer:
    """
    Calculate comprehensive performance metrics.

    Uses realistic annualization (252 trading days) and proper
    risk-adjusted metrics.
    """

    @staticmethod
    def calculate_metrics(results: Dict, risk_free_rate: float = 0.02) -> Dict:
        """
        Calculate all performance metrics with PROPER SHARPE RATIO.

        CRITICAL IMPROVEMENT: Correct Sharpe Ratio Formula
        ==================================================
        Sharpe Ratio = (Return - Risk_Free_Rate) / Volatility

        Previous implementations often omit risk-free rate or use 0.
        This improvement uses actual three-month Treasury yield rates for accuracy.

        METRICS INCLUDE:
        - Total return and CAGR
        - Volatility (annualized)
        - Sharpe ratio (with risk-free rate)
        - Sortino ratio (downside risk-adjusted)
        - Calmar ratio (return / max drawdown)
        - Maximum drawdown
        - Hit ratio (% positive trades)
        - Alpha vs benchmark (risk-free adjusted)

        Parameters:
        -----------
        results : Dict
            Backtest results
        risk_free_rate : float
            Annual risk-free rate (default: 2%)

        Returns:
        --------
        Dict with all metrics
        """
        df = results["data"]
        trades = results["trades"]

        # Strategy metrics
        strategy_returns = df["Strategy_Returns"].dropna()
        n_years = len(strategy_returns) / 252

        final_value = df["Portfolio_Value"].iloc[-1]
        initial_value = results["initial_capital"]

        total_return = (final_value / initial_value) - 1
        cagr = (final_value / initial_value) ** (1/n_years) - 1 if n_years > 0 else 0

        volatility = strategy_returns.std() * np.sqrt(252)

        # CORRECT SHARPE: (Return - RiskFree) / Volatility
        sharpe = ((cagr - risk_free_rate) / volatility) if volatility > 0 else 0

        # Sortino ratio (only downside volatility)
        downside_returns = strategy_returns[strategy_returns < 0]
        downside_vol = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else volatility
        sortino = ((cagr - risk_free_rate) / downside_vol) if downside_vol > 0 else 0

        # Maximum drawdown
        cumulative = (1 + strategy_returns).cumprod()
        running_max = cumulative.expanding().max()
        drawdown = (cumulative - running_max) / running_max
        max_dd = drawdown.min()

        # Calmar ratio
        calmar = ((cagr - risk_free_rate) / abs(max_dd)) if max_dd != 0 else 0

        # Hit ratio (% of positive trades)
        if len(trades) >= 2:
            # Calculate P&L for each round trip
            buy_trades = [t for t in trades if t["Action"] == "BUY"]
            sell_trades = [t for t in trades if t["Action"] == "SELL"]

            wins = 0
            total_trades = min(len(buy_trades), len(sell_trades))

            for i in range(total_trades):
                buy_price = buy_trades[i]["Price"]
                sell_price = sell_trades[i]["Price"]
                if sell_price > buy_price:
                    wins += 1

            hit_ratio = wins / total_trades if total_trades > 0 else 0
        else:
            hit_ratio = 0

        # Benchmark metrics
        bench_returns = df["Bench_Returns"].dropna()
        bench_final = df["Bench_Value"].iloc[-1]

        bench_total_return = (bench_final / initial_value) - 1
        bench_cagr = (bench_final / initial_value) ** (1/n_years) - 1 if n_years > 0 else 0
        bench_volatility = bench_returns.std() * np.sqrt(252)

        # Benchmark Sharpe (with risk-free rate)
        bench_sharpe = ((bench_cagr - risk_free_rate) / bench_volatility) if bench_volatility > 0 else 0

        # Calculate benchmark drawdown
        bench_cumulative = (1 + bench_returns).cumprod()
        bench_running_max = bench_cumulative.expanding().max()
        bench_drawdown = (bench_cumulative - bench_running_max) / bench_running_max
        bench_max_dd = bench_drawdown.min()

        # Alpha (excess return vs benchmark)
        alpha = cagr - bench_cagr

        # Information ratio (alpha / tracking error)
        tracking_error = (strategy_returns - bench_returns).std() * np.sqrt(252)
        info_ratio = (alpha / tracking_error) if tracking_error > 0 else 0

        metrics = {
            "Strategy": {
                "Total Return": total_return,
                "CAGR": cagr,
                "Volatility": volatility,
                "Sharpe Ratio": sharpe,
                "Sortino Ratio": sortino,
                "Calmar Ratio": calmar,
                "Max Drawdown": max_dd,
                "Hit Ratio": hit_ratio,
                "Total Trades": len(trades),
                "Final Value": final_value
            },
            "Benchmark": {
                "Total Return": bench_total_return,
                "CAGR": bench_cagr,
                "Volatility": bench_volatility,
                "Sharpe Ratio": bench_sharpe,
                "Max Drawdown": bench_max_dd,
                "Final Value": bench_final
            },
            "Alpha": alpha,
            "Information Ratio": info_ratio,
            "Risk_Free_Rate": risk_free_rate
        }

        return metrics

    @staticmethod
    def print_metrics(metrics: Dict):
        """Print formatted metrics table with all improvements."""
        print("\n" + "="*80)
        print("PERFORMANCE METRICS")
        print("="*80)

        print(f"\nRisk-Free Rate (Three-Month Treasury Yield): {metrics['Risk_Free_Rate']:.2%}")

        print("\nSTRATEGY PERFORMANCE:")
        for name, value in metrics["Strategy"].items():
            if name in ["Total Return", "CAGR", "Volatility", "Max Drawdown"]:
                print(f"  {name:20s}: {value:>10.2%}")
            elif name in ["Sharpe Ratio", "Sortino Ratio", "Calmar Ratio", "Hit Ratio"]:
                print(f"  {name:20s}: {value:>10.2f}")
            elif name == "Total Trades":
                print(f"  {name:20s}: {value:>10d}")
            else:
                print(f"  {name:20s}: ${value:>10,.2f}")

        print("\nBENCHMARK PERFORMANCE:")
        for name, value in metrics["Benchmark"].items():
            if name in ["Total Return", "CAGR", "Volatility", "Max Drawdown"]:
                print(f"  {name:20s}: {value:>10.2%}")
            elif name in ["Sharpe Ratio"]:
                print(f"  {name:20s}: {value:>10.2f}")
            else:
                print(f"  {name:20s}: ${value:>10,.2f}")

        print(f"\nEXCESS PERFORMANCE:")
        print(f"  Alpha (vs Benchmark): {metrics['Alpha']:+.2%}")
        print(f"  Information Ratio:    {metrics['Information Ratio']:>10.2f}")
        print("="*80 + "\n")


# ============================================================================
# VISUALIZATION
# ============================================================================

def create_visualizations(results: Dict, config: StrategyConfig):
    """
    Create comprehensive visualization dashboard.

    PLOTS:
    1. Equity curve (strategy vs benchmark)
    2. Drawdown curve
    3. Rolling 6-month Sharpe ratio
    4. VaR vs actual returns scatter
    """
    df = results["data"]

    fig, axes = plt.subplots(4, 1, figsize=(16, 14))

    # === PLOT 1: EQUITY CURVE ===
    ax1 = axes[0]
    ax1.plot(df["Date"], df["Portfolio_Value"],
            label="Strategy", linewidth=2.5, color="darkblue")
    ax1.plot(df["Date"], df["Bench_Value"],
            label=f"{config.BENCHMARK} (Benchmark)",
            linewidth=2, color="orange", alpha=0.7)

    ax1.set_title("Equity Curve: Strategy vs Benchmark",
                 fontsize=14, fontweight="bold")
    ax1.set_ylabel("Portfolio Value ($)", fontsize=12)
    ax1.legend(loc="upper left", fontsize=11)
    ax1.grid(True, alpha=0.3)

    # === PLOT 2: DRAWDOWN ===
    ax2 = axes[1]

    # Calculate drawdowns
    strategy_cumulative = (1 + df["Strategy_Returns"]).cumprod()
    strategy_running_max = strategy_cumulative.expanding().max()
    strategy_dd = (strategy_cumulative - strategy_running_max) / strategy_running_max

    bench_cumulative = (1 + df["Bench_Returns"]).cumprod()
    bench_running_max = bench_cumulative.expanding().max()
    bench_dd = (bench_cumulative - bench_running_max) / bench_running_max

    ax2.fill_between(df["Date"], strategy_dd * 100, 0,
                     alpha=0.7, color="red", label="Strategy DD")
    ax2.fill_between(df["Date"], bench_dd * 100, 0,
                     alpha=0.5, color="orange", label="Benchmark DD")

    ax2.set_title("Drawdown Analysis", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Drawdown (%)", fontsize=12)
    ax2.legend(loc="lower left", fontsize=11)
    ax2.grid(True, alpha=0.3)

    # === PLOT 3: ROLLING SHARPE ===
    ax3 = axes[2]

    # Calculate rolling Sharpe
    rolling_window = config.ROLLING_SHARPE_WINDOW
    rolling_mean = df["Strategy_Returns"].rolling(rolling_window).mean() * 252
    rolling_std = df["Strategy_Returns"].rolling(rolling_window).std() * np.sqrt(252)
    rolling_sharpe = rolling_mean / rolling_std

    ax3.plot(df["Date"], rolling_sharpe,
            linewidth=2, color="green", label=f"{rolling_window}-day Rolling Sharpe")
    ax3.axhline(y=0, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax3.axhline(y=1, color="gray", linestyle=":", linewidth=1, alpha=0.5)

    ax3.set_title("Rolling Sharpe Ratio (6-month)",
                 fontsize=14, fontweight="bold")
    ax3.set_ylabel("Sharpe Ratio", fontsize=12)
    ax3.legend(loc="upper left", fontsize=11)
    ax3.grid(True, alpha=0.3)

    # === PLOT 4: VaR vs ACTUAL RETURNS ===
    ax4 = axes[3]

    # Filter out NaN values
    valid_data = df[["VaR_Lag", "LogReturns"]].dropna()

    ax4.scatter(valid_data["VaR_Lag"] * 100,
               -valid_data["LogReturns"] * 100,
               alpha=0.5, s=20, color="purple")

    # Add reference line (45-degree)
    max_var = valid_data["VaR_Lag"].max() * 100
    ax4.plot([0, max_var], [0, max_var],
            "r--", linewidth=2, alpha=0.7, label="VaR = Actual Loss")

    # Add threshold line
    ax4.axvline(x=config.VAR_THRESHOLD * 100,
               color="green", linestyle=":", linewidth=2,
               label=f"VaR Threshold ({config.VAR_THRESHOLD:.1%})")

    ax4.set_title("VaR Filter: Predicted vs Actual Losses",
                 fontsize=14, fontweight="bold")
    ax4.set_xlabel("VaR (10th percentile) - %", fontsize=12)
    ax4.set_ylabel("Actual Loss - %", fontsize=12)
    ax4.legend(loc="upper left", fontsize=11)
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """
    Main execution function with all improvements.

    COMPREHENSIVE ANALYSIS PIPELINE:
    ================================
    1. Fetch adjusted price data (accounts for dividends/splits)
    2. Fetch risk-free rate (three-month Treasury yields)
    3. Split data: Train (70%) / Test (30%)
    4. Run backtest on training data
    5. Run backtest on test data (out-of-sample validation)
    6. Compare performance (detect overfitting)
    7. Parameter robustness analysis (27 combinations)
    8. Generate comprehensive visualizations
    """

    print("="*80)
    print("VOLATILITY-ADJUSTED MEAN REVERSION STRATEGY")
    print("WITH COMPREHENSIVE VALIDATION")
    print("="*80)

    # Initialize configuration
    config = StrategyConfig()

    print(f"\nCONFIGURATION:")
    print(f"  Ticker: {config.TICKER} (Nasdaq-100 ETF)")
    print(f"  Benchmark: {config.BENCHMARK} (S&P 500 ETF)")
    print(f"  Period: {config.START_DATE} to {config.END_DATE}")
    print(f"  Train/Test Split: {config.TRAIN_END_DATE}")
    print(f"  Initial Capital: ${config.INITIAL_CAPITAL:,.0f}")
    print(f"  Commission: ${config.COMMISSION_PER_TRADE} per trade")

    print(f"\n  DEFAULT PARAMETERS:")
    print(f"    Z-Score Window: {config.ZSCORE_WINDOW} days")
    print(f"    Z-Score Threshold: {config.ZSCORE_THRESHOLD}")
    print(f"    VaR Window: {config.VAR_WINDOW} days")
    print(f"    VaR Percentile: {config.VAR_PERCENTILE}%")
    print(f"    VaR Threshold: {config.VAR_THRESHOLD:.1%}")

    # === STEP 1: DATA ACQUISITION (ADJUSTED PRICES) ===
    print("\n" + "-"*80)
    print("STEP 1: FETCHING ADJUSTED PRICE DATA")
    print("-"*80)

    fetcher = DataFetcher()

    # Fetch trading data (QQQ)
    ticker_data = fetcher.fetch_historical_data(
        config.TICKER, config.START_DATE, config.END_DATE
    )

    # Fetch benchmark data (SPY)
    benchmark_data = fetcher.fetch_historical_data(
        config.BENCHMARK, config.START_DATE, config.END_DATE
    )

    # Fetch risk-free rate
    rf_data = fetcher.fetch_risk_free_rate(config.START_DATE, config.TRAIN_END_DATE)
    config.RISK_FREE_RATE = float(rf_data["rf_rate"].mean())
    print(f"  Using the training period's mean three-month Treasury yield: {config.RISK_FREE_RATE:.2%}")

    # === STEP 2: OUT-OF-SAMPLE DATA SPLIT ===
    print("\n" + "-"*80)
    print("STEP 2: OUT-OF-SAMPLE VALIDATION SETUP")
    print("-"*80)

    validator = OutOfSampleValidator()

    # Prepare features on full dataset first
    data_with_features = FeatureEngine.prepare_features(ticker_data, config)

    # Split into train and test
    train_data, test_data = validator.split_data(
        data_with_features, config.TRAIN_END_DATE
    )

    # === STEP 3: IN-SAMPLE (TRAINING) BACKTEST ===
    print("\n" + "-"*80)
    print("STEP 3: IN-SAMPLE (TRAINING) BACKTEST")
    print("-"*80)

    # Generate signals on training data
    signal_gen_train = SignalGenerator(config)
    train_with_signals = signal_gen_train.generate_signals(train_data)

    num_train_signals = (train_with_signals["Signal"] == 1).sum()
    print(f"Training period long signals: {num_train_signals}")

    # Run training backtest
    engine_train = BacktestEngine(config)
    results_train = engine_train.run_backtest(train_with_signals)

    # Add benchmark to training results
    benchmark_train = benchmark_data[
        benchmark_data["Date"] <= pd.to_datetime(config.TRAIN_END_DATE)
    ]
    results_train = engine_train.add_benchmark(results_train, benchmark_train)

    # Calculate training metrics
    metrics_train = PerformanceAnalyzer.calculate_metrics(
        results_train, config.RISK_FREE_RATE
    )

    print("\nTRAINING PERFORMANCE:")
    PerformanceAnalyzer.print_metrics(metrics_train)

    # === STEP 4: OUT-OF-SAMPLE (TEST) BACKTEST ===
    print("\n" + "-"*80)
    print("STEP 4: OUT-OF-SAMPLE (TEST) BACKTEST")
    print("-"*80)

    # Generate signals on test data
    signal_gen_test = SignalGenerator(config)
    test_with_signals = signal_gen_test.generate_signals(test_data)

    num_test_signals = (test_with_signals["Signal"] == 1).sum()
    print(f"Test period long signals: {num_test_signals}")

    # Run test backtest
    engine_test = BacktestEngine(config)
    results_test = engine_test.run_backtest(test_with_signals)

    # Add benchmark to test results
    benchmark_test = benchmark_data[
        benchmark_data["Date"] > pd.to_datetime(config.TRAIN_END_DATE)
    ]
    results_test = engine_test.add_benchmark(results_test, benchmark_test)

    # Calculate test metrics
    metrics_test = PerformanceAnalyzer.calculate_metrics(
        results_test, config.RISK_FREE_RATE
    )

    print("\nTEST PERFORMANCE:")
    PerformanceAnalyzer.print_metrics(metrics_test)

    # === STEP 5: COMPARE IN-SAMPLE VS OUT-OF-SAMPLE ===
    validator.compare_performance(metrics_train, metrics_test)

    # === STEP 6: ROBUSTNESS ANALYSIS ===
    print("\n" + "-"*80)
    print("STEP 6: PARAMETER ROBUSTNESS ANALYSIS")
    print("-"*80)
    print("\nTesting 27 parameter combinations...")
    print("This may take a few minutes...")

    analyzer = RobustnessAnalyzer(config)

    # Run robustness analysis on TRAINING data only
    # (Never use test data for parameter selection)
    robustness_results = analyzer.parameter_sensitivity_analysis(
        train_data, benchmark_train
    )

    # === STEP 7: VISUALIZATIONS ===
    print("\n" + "-"*80)
    print("STEP 7: GENERATING VISUALIZATIONS")
    print("-"*80)

    # Training period visualization
    print("\nCreating training period charts...")
    create_visualizations(results_train, config)

    # Test period visualization
    print("\nCreating test period (out-of-sample) charts...")
    create_visualizations(results_test, config)

    # Robustness heatmaps
    print("\nCreating parameter sensitivity heatmaps...")
    analyzer.visualize_parameter_sensitivity(robustness_results)

    # === STEP 8: FINAL SUMMARY ===
    print("\n" + "="*80)
    print("FINAL SUMMARY - STRATEGY VALIDATION")
    print("="*80)

    print(f"\nSTRATEGY: Volatility-Adjusted Mean Reversion (VaR-Filtered)")
    print(f"TICKER: {config.TICKER} (Nasdaq-100)")
    print(f"BENCHMARK: {config.BENCHMARK} (S&P 500)")
    print(f"PERIOD: {config.START_DATE} to {config.END_DATE}")

    print(f"\n1. ADJUSTED PRICES:")
    print(f"   ✓ Uses dividend-adjusted returns")
    print(f"   ✓ Accounts for stock splits")
    print(f"   ✓ Reflects true total returns")

    print(f"\n2. PROPER SHARPE RATIO:")
    print(f"   ✓ Risk-free rate: {config.RISK_FREE_RATE:.2%}")
    print(f"   ✓ Formula: (Return - RiskFree) / Volatility")

    print(f"\n3. OUT-OF-SAMPLE VALIDATION:")
    print(f"   Training Sharpe: {metrics_train['Strategy']['Sharpe Ratio']:.2f}")
    print(f"   Test Sharpe: {metrics_test['Strategy']['Sharpe Ratio']:.2f}")

    sharpe_degradation = ((metrics_test['Strategy']['Sharpe Ratio'] -
                          metrics_train['Strategy']['Sharpe Ratio']) /
                         abs(metrics_train['Strategy']['Sharpe Ratio']))

    if metrics_test['Strategy']['Sharpe Ratio'] > 0:
        print(f"   ✓ Positive out-of-sample Sharpe")
    else:
        print(f"   ✗ Negative out-of-sample Sharpe - NO PREDICTIVE POWER")

    if sharpe_degradation > -0.3:
        print(f"   ✓ Performance holds up well ({sharpe_degradation:+.1%} change)")
    else:
        print(f"   ⚠ Significant degradation ({sharpe_degradation:+.1%} change)")

    print(f"\n4. ROBUSTNESS ANALYSIS:")
    print(f"   Mean Sharpe: {robustness_results['Sharpe'].mean():.2f}")
    print(f"   Std Sharpe: {robustness_results['Sharpe'].std():.2f}")
    print(f"   Positive Sharpe combos: {(robustness_results['Sharpe'] > 0).sum()}/27")

    cv = robustness_results['Sharpe'].std() / robustness_results['Sharpe'].mean()
    if cv < 0.5:
        print(f"   ✓ Low parameter sensitivity (CV={cv:.2f})")
    else:
        print(f"   ⚠ High parameter sensitivity (CV={cv:.2f})")

    print(f"\n5. RECOMMENDATION:")

    # Decision logic
    if (metrics_test['Strategy']['Sharpe Ratio'] > 0 and
        sharpe_degradation > -0.5 and
        cv < 0.6):
        print(f"   ✓ APPROVED FOR LIVE TESTING")
        print(f"     Strategy shows:")
        print(f"     - Positive out-of-sample performance")
        print(f"     - Reasonable robustness")
        print(f"     - Not severely overfit")
        print(f"\n   Next steps:")
        print(f"     1. Paper trade for 1-3 months")
        print(f"     2. Monitor slippage and execution")
        print(f"     3. Verify signal generation in real-time")
    else:
        print(f"   ⚠ NOT RECOMMENDED FOR LIVE TRADING")
        print(f"     Issues detected:")
        if metrics_test['Strategy']['Sharpe Ratio'] <= 0:
            print(f"     - No out-of-sample predictive power")
        if sharpe_degradation <= -0.5:
            print(f"     - Severe overfitting")
        if cv >= 0.6:
            print(f"     - High parameter sensitivity")
        print(f"\n   Recommendations:")
        print(f"     1. Revise strategy logic")
        print(f"     2. Test alternative parameters")
        print(f"     3. Consider different assets or timeframes")

    print("\n" + "="*80)
    print("ANALYSIS COMPLETE")
    print("="*80)

    return {
        "train_results": results_train,
        "test_results": results_test,
        "train_metrics": metrics_train,
        "test_metrics": metrics_test,
        "robustness_results": robustness_results,
        "config": config
    }


if __name__ == "__main__":
    all_results = main()
